# Pokémon TCG: initial EDA

We started by checking what a row represents, how prices differ by rarity, and where data is missing.

## Research questions

1. Can historical pricing trends and card characteristics predict whether an individual Pokémon card's value will increase or decrease?
2. Which Pokémon card characteristics, such as rarity, age, and grading condition, are most strongly associated with market value?
3. Have individual Pokémon card prices increased faster than general consumer inflation?

This notebook is the initial exploration, not a prediction model. Inflation is still future work and needs economic data for matching months and currency.

The main clustering analysis follows in `Clustering_Characteristics.ipynb`.

# 1. What does a row represent?

Load the combined table and check its size, date range, and a few examples.

In [1]:
from main import load_data

df = load_data()

print("Shape:", df.shape)
print("Products:", df["pc_id"].nunique())
print("Cards:", df["card_id"].nunique())
print("Months:", df["month"].min(), "to", df["month"].max())

display(
    df[["card_id", "pc_id", "name", "variant", "month", "price_ungraded", "price_psa10"]].head()
)

Shape: (1241631, 33)
Products: 30165
Cards: 16812
Months: 2020-12 to 2024-11


,card_id,pc_id,name,variant,month,price_ungraded,price_psa10
0,base1-1,715590,Alakazam,1st-edition,2020-12,NaN,NaN
1,base1-1,715590,Alakazam,1st-edition,2021-01,220.20,NaN
2,base1-1,715590,Alakazam,1st-edition,2021-02,246.68,10400.0
3,base1-1,715590,Alakazam,1st-edition,2021-03,352.50,10400.0
4,base1-1,715590,Alakazam,1st-edition,2021-04,375.00,10400.0


## Observations

A row is one product in one month. `card_id` identifies a card, while `pc_id` identifies a particular printing. One card can have several printings, and each product can appear in many months, so the row count isn't a count of independent cards.

Prices are in USD. The loader has already converted cents to dollars and source price zeros to missing values. This linked table isn't the untouched source and doesn't include every source product.

# 2. Prices by rarity

Start with November 2024, the latest month. Each product contributes at most once, rather than getting more weight for having a longer history. The summaries use recorded prices.

In [3]:
november = df[df["month"] == "2024-11"]
rarities = ["Common", "Uncommon", "Rare", "Rare Holo", "Rare Ultra"]

prices_by_rarity = november.groupby("rarity")["price_ungraded"].agg(
    products_with_price="count",
    median="median",
    lower_quartile=lambda prices: prices.quantile(0.25),
    upper_quartile=lambda prices: prices.quantile(0.75),
)

display(prices_by_rarity.loc[rarities].round(2))

,products_with_price,median,lower_quartile,upper_quartile
rarity,,,,
Common,8738,1.48,1.15,2.30
Uncommon,8473,1.63,1.25,2.70
Rare,4748,2.05,1.49,4.44
Rare Holo,3342,3.49,1.97,11.55
Rare Ultra,795,7.56,3.90,18.50


## Observations

The median rises across these selected categories, from 1.48 USD for Common to 7.56 USD for Rare Ultra. Rare Holo and Rare Ultra also have wider middle ranges, but the ranges overlap. Rarity alone doesn't determine price.

We use the median here so a few very expensive products don't pull the summary upward as much as they would with the mean.

## Common cards: ungraded versus PSA 10

Grading costs money, so people may submit cards they expect to be worth more after grading. As a first check, compare Common products with both prices recorded for the same printing and month.

In [4]:
common = november[november["rarity"] == "Common"]
paired = common.dropna(subset=["price_ungraded", "price_psa10"])

price_ratio = paired["price_psa10"] / paired["price_ungraded"]

print("Common products with both prices:", len(paired))
print("Median PSA 10 / ungraded price:", round(price_ratio.median(), 2))

Common products with both prices: 5767
Median PSA 10 / ungraded price: 26.78


## Observation

The median of the per-product PSA 10/ungraded ratios is 26.78. That's a price comparison for the matched subset, not grading profit. The grade a submitted copy would receive, costs, and fees aren't included; products missing either price aren't compared.

# 3. Missing prices and missing rows

A row can exist without a PSA 10 price, or a product can have no row that month. We count both against the IDs in `card_links.csv`, first checking whether any linked product is absent from the whole panel.

In [4]:
import pandas as pd

links = pd.read_csv("data/card_links.csv", dtype={"pc_id": str}, keep_default_na=False)
linked_products = set(links["pc_id"])
panel_products = set(df["pc_id"].astype(str))

print("Linked products:", len(linked_products))
print("Linked products absent from the whole panel:", len(linked_products - panel_products))
print("Panel products absent from links:", len(panel_products - linked_products))

total_products = len(linked_products)
linked_panel = df[df["pc_id"].astype(str).isin(linked_products)]

coverage = linked_panel.groupby("month").agg(
    rows_present=("pc_id", "size"),
    psa10_present=("price_psa10", "count"),
)

coverage["row_present_psa10_missing"] = coverage["rows_present"] - coverage["psa10_present"]
coverage["no_row"] = total_products - coverage["rows_present"]
coverage["psa10_pct_of_present_rows"] = 100 * coverage["psa10_present"] / coverage["rows_present"]

print("All linked products:", total_products)
display(coverage.loc["2024-07":"2024-11"].round(1))

Linked products: 30165
Linked products absent from the whole panel: 0
Panel products absent from links: 0
All linked products: 30165


,rows_present,psa10_present,row_present_psa10_missing,no_row,psa10_pct_of_present_rows
month,,,,,
2024-07,30041,22177,7864,124,73.8
2024-08,30086,22407,7679,79,74.5
2024-09,30142,22333,7809,23,74.1
2024-10,18118,13751,4367,12047,75.9
2024-11,30091,22494,7597,74,74.8


## Observations

All 30,165 linked products appear somewhere in the panel, but not in every month.

October 2024 has many fewer product rows than the surrounding months. A percentage calculated only from present rows hides those absent products. We don't treat missing records as zero prices or fill the gap.

`Clustering_History_Interpretation.ipynb` follows up with monthly coverage and price comparisons over the available history.